In [1]:
# generate_data.py
import pandas as pd
import numpy as np

np.random.seed(42)
n_customers = 1000
n_days = 30

records = []
for cust_id in range(n_customers):
    for day in range(n_days):
        records.append({
            "customer_id": cust_id,
            "day": day,
            "pages_viewed": np.random.poisson(3),
            "time_spent_min": round(np.random.exponential(5), 2),
            "items_in_cart": np.random.randint(0, 6),
            "discount_clicked": np.random.binomial(1, 0.3),
            "returned_visit": np.random.binomial(1, 0.4),
        })

df = pd.DataFrame(records)

# Label: did this customer buy in the next 7 days?
# Customers with high cart items + time spent are more likely to buy

df = df.sort_values(["customer_id", "day"]).reset_index(drop=True)

# Rolling 3-day average of cart + time spent, per customer
df["cart_trend"] = df.groupby("customer_id")["items_in_cart"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df["time_trend"] = df.groupby("customer_id")["time_spent_min"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

score = (
    df["cart_trend"] * 0.45 +
    df["time_trend"] * 0.12 +
    df["discount_clicked"] * 0.25 +
    df["returned_visit"] * 0.18
)

# Use median as threshold so classes stay balanced
df["will_buy"] = (score > score.median()).astype(int)

df.to_csv("customer_data.csv", index=False)
print("Dataset created:", df.shape)

Dataset created: (30000, 10)
